In [4]:
import os
import h5py
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# CONFIG
# =========================================================
ROOT = "/content/drive/MyDrive/V2_drift/features/pyramid_texture"
SUBJECT = 8 # change per subject

file1 = os.path.join(
    ROOT,
    f"S{SUBJECT}_features_each_prf_4ori_4sf_grid5_npix256.h5py"
)

file2 = os.path.join(
    ROOT,
    f"S{SUBJECT}_features_each_prf_4ori_4sf_grid5_npix256_second.h5py"
)

output_file = os.path.join(
    ROOT,
    f"S{SUBJECT}_features_each_prf_4ori_4sf_grid5_npix256_merged.h5py"
)

# =========================================================
# MERGE FUNCTION
# =========================================================
def merge_feature_batches(file1, file2, output_file, dataset_name="features"):
    print("Merging files:")
    print("  file1:", file1)
    print("  file2:", file2)
    print("  output:", output_file)

    if not os.path.exists(file1):
        raise FileNotFoundError(f"First file not found: {file1}")
    if not os.path.exists(file2):
        raise FileNotFoundError(f"Second file not found: {file2}")

    with h5py.File(file1, "r") as f1, h5py.File(file2, "r") as f2:
        if dataset_name not in f1:
            raise KeyError(f"Dataset '{dataset_name}' not found in {file1}")
        if dataset_name not in f2:
            raise KeyError(f"Dataset '{dataset_name}' not found in {file2}")

        d1 = f1[dataset_name]
        d2 = f2[dataset_name]

        print("\nInput shapes:")
        print("  file1:", d1.shape)
        print("  file2:", d2.shape)

        # Expect shape: (n_images, n_features, n_prfs)
        if len(d1.shape) != 3 or len(d2.shape) != 3:
            raise ValueError("Expected 3D feature arrays: (n_images, n_features, n_prfs)")

        # Features/prfs must match
        if d1.shape[1:] != d2.shape[1:]:
            raise ValueError(
                f"Feature dimensions mismatch:\n"
                f"  file1: {d1.shape}\n"
                f"  file2: {d2.shape}"
            )

        n1 = d1.shape[0]
        n2 = d2.shape[0]
        total_n = n1 + n2

        print("\nCreating merged file...")
        with h5py.File(output_file, "w") as fout:
            dset_out = fout.create_dataset(
                dataset_name,
                shape=(total_n, d1.shape[1], d1.shape[2]),
                dtype=d1.dtype,
                compression="gzip"
            )

            print("Writing first batch...")
            dset_out[:n1, :, :] = d1[:, :, :]

            print("Writing second batch...")
            dset_out[n1:n1+n2, :, :] = d2[:, :, :]

    print("\nMerge complete.")
    print("Saved to:", output_file)

# =========================================================
# RUN MERGE
# =========================================================
merge_feature_batches(file1, file2, output_file)

# =========================================================
# VERIFY OUTPUT
# =========================================================
print("\nVerifying merged file...")
with h5py.File(file1, "r") as f1, h5py.File(file2, "r") as f2, h5py.File(output_file, "r") as fout:
    d1 = f1["features"]
    d2 = f2["features"]
    dout = fout["features"]

    print("file1 shape :", d1.shape)
    print("file2 shape :", d2.shape)
    print("merged shape:", dout.shape)

    expected_shape = (d1.shape[0] + d2.shape[0], d1.shape[1], d1.shape[2])
    print("expected    :", expected_shape)

    assert dout.shape == expected_shape, "Merged shape is wrong."

    # Check first and last entries to confirm order
    print("\nChecking content consistency...")
    first_ok = np.allclose(dout[0], d1[0], equal_nan=True)
    split_ok = np.allclose(dout[d1.shape[0] - 1], d1[d1.shape[0] - 1], equal_nan=True)
    second_start_ok = np.allclose(dout[d1.shape[0]], d2[0], equal_nan=True)
    last_ok = np.allclose(dout[-1], d2[-1], equal_nan=True)

    print("first sample matches file1[0]:", first_ok)
    print("last sample of first batch matches:", split_ok)
    print("first sample of second batch matches:", second_start_ok)
    print("last sample matches file2[-1]:", last_ok)

    assert first_ok, "First sample mismatch."
    assert split_ok, "End of first batch mismatch."
    assert second_start_ok, "Start of second batch mismatch."
    assert last_ok, "Last sample mismatch."

print("\n✅ Merge verified successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Merging files:
  file1: /content/drive/MyDrive/V2_drift/features/pyramid_texture/S8_features_each_prf_4ori_4sf_grid5_npix256.h5py
  file2: /content/drive/MyDrive/V2_drift/features/pyramid_texture/S8_features_each_prf_4ori_4sf_grid5_npix256_second.h5py
  output: /content/drive/MyDrive/V2_drift/features/pyramid_texture/S8_features_each_prf_4ori_4sf_grid5_npix256_merged.h5py

Input shapes:
  file1: (10000, 641, 1456)
  file2: (10000, 641, 1456)

Creating merged file...
Writing first batch...
Writing second batch...

Merge complete.
Saved to: /content/drive/MyDrive/V2_drift/features/pyramid_texture/S8_features_each_prf_4ori_4sf_grid5_npix256_merged.h5py

Verifying merged file...
file1 shape : (10000, 641, 1456)
file2 shape : (10000, 641, 1456)
merged shape: (20000, 641, 1456)
expected    : (20000, 641, 1456)

Checking content consistency...
first sample matches f

# replace the original first-batch filename with the merged one, if your loader expects that exact name.

In [ ]:
import h5py
import numpy as np
import os
from google.colab import drive
drive.mount('/content/drive')


ROOT = "/content/drive/MyDrive/V2_drift/features/pyramid_texture"
SUBJECT = 3

file1 = os.path.join(ROOT, f"S{SUBJECT}_features_each_prf_4ori_4sf_grid5_npix256_first.h5py")
file2 = os.path.join(ROOT, f"S{SUBJECT}_features_each_prf_4ori_4sf_grid5_npix256_second.h5py")
merged = os.path.join(ROOT, f"S{SUBJECT}_features_each_prf_4ori_4sf_grid5_npix256_merged.h5py")

print("Checking merged file:", merged)

with h5py.File(file1, "r") as f1, h5py.File(file2, "r") as f2, h5py.File(merged, "r") as fm:

    d1 = f1["features"]
    d2 = f2["features"]
    dm = fm["features"]

    print("\n=== SHAPES ===")
    print("file1:", d1.shape)
    print("file2:", d2.shape)
    print("merged:", dm.shape)

    expected = (d1.shape[0] + d2.shape[0], d1.shape[1], d1.shape[2])
    print("expected:", expected)

    assert dm.shape == expected, "❌ Shape mismatch!"

    print("\n=== BASIC VALUE CHECK ===")

    # Check boundaries
    print("Checking first element...")
    print(np.allclose(dm[0], d1[0]))

    print("Checking last element of first batch...")
    print(np.allclose(dm[d1.shape[0] - 1], d1[d1.shape[0] - 1]))

    print("Checking first element of second batch...")
    print(np.allclose(dm[d1.shape[0]], d2[0]))

    print("Checking last element...")
    print(np.allclose(dm[-1], d2[-1]))

    print("\n=== RANDOM SAMPLE CHECK ===")

    # random checks
    for i in [10, 100, 500]:
        if i < d1.shape[0]:
            ok = np.allclose(dm[i], d1[i])
            print(f"Check file1 index {i}: {ok}")

    for i in [10, 100, 500]:
        if i < d2.shape[0]:
            ok = np.allclose(dm[d1.shape[0] + i], d2[i])
            print(f"Check file2 index {i}: {ok}")

    print("\n=== VALUE STATS ===")
    sample = dm[0:10]  # small sample
    print("mean:", np.mean(sample))
    print("std :", np.std(sample))
    print("min :", np.min(sample))
    print("max :", np.max(sample))

print("\n Merge looks correct!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking merged file: /content/drive/MyDrive/V2_drift/features/pyramid_texture/S3_features_each_prf_4ori_4sf_grid5_npix256_merged.h5py

=== SHAPES ===
file1: (10000, 641, 1456)
file2: (10000, 641, 1456)
merged: (20000, 641, 1456)
expected: (20000, 641, 1456)

=== BASIC VALUE CHECK ===
Checking first element...
True
Checking last element of first batch...
True
Checking first element of second batch...
True
Checking last element...
True

=== RANDOM SAMPLE CHECK ===
Check file1 index 10: True
Check file1 index 100: True
Check file1 index 500: True
Check file2 index 10: True
Check file2 index 100: True
Check file2 index 500: True

=== VALUE STATS ===
mean: 9.45305
std : 138.7622
min : -1750.1346
max : 143752.3

✅ Merge looks correct!


# addition of first and second batches files.

In [ ]:
import os
import gc
import h5py
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

ROOT = "/content/drive/MyDrive/V2_drift/features/pyramid_texture"
SUBJECT = 4   # change per subject
CHUNK_SIZE = 1000

file1 = os.path.join(
    ROOT,
    f"S{SUBJECT}_features_each_prf_4ori_4sf_grid5_npix256.h5py"
)

file2 = os.path.join(
    ROOT,
    f"S{SUBJECT}_features_each_prf_4ori_4sf_grid5_npix256_second.h5py"
)

output_file = os.path.join(
    ROOT,
    f"S{SUBJECT}_features_each_prf_4ori_4sf_grid5_npix256_merged.h5py"
)

def merge_overlay(file1, file2, output_file, dataset_name="features", chunk_size=100):
    with h5py.File(file1, "r") as f1, h5py.File(file2, "r") as f2:
        d1 = f1[dataset_name]
        d2 = f2[dataset_name]

        print("file1 shape:", d1.shape)
        print("file2 shape:", d2.shape)

        assert d1.shape == d2.shape, "Shapes must match"
        n_img, n_feat, n_prf = d1.shape

        with h5py.File(output_file, "w") as fout:
            dout = fout.create_dataset(
                dataset_name,
                shape=(n_img, n_feat, n_prf),
                dtype=d1.dtype,
                compression="gzip"
            )

            for start in range(0, n_img, chunk_size):
                end = min(start + chunk_size, n_img)
                print(f"Processing rows {start}:{end}")

                c1 = np.array(d1[start:end], dtype=d1.dtype)
                c2 = np.array(d2[start:end], dtype=d2.dtype)

                # Since missing half is stored as zeros, combine by addition.
                merged = c1 + c2

                dout[start:end] = merged

                del c1, c2, merged
                gc.collect()

    print("Saved merged file to:", output_file)

merge_overlay(file1, file2, output_file, chunk_size=CHUNK_SIZE)

Mounted at /content/drive
file1 shape: (10000, 641, 1456)
file2 shape: (10000, 641, 1456)
Processing rows 0:1000
Processing rows 1000:2000
Processing rows 2000:3000
Processing rows 3000:4000
Processing rows 4000:5000
Processing rows 5000:6000
Processing rows 6000:7000
Processing rows 7000:8000
Processing rows 8000:9000
Processing rows 9000:10000
Saved merged file to: /content/drive/MyDrive/V2_drift/features/pyramid_texture/S4_features_each_prf_4ori_4sf_grid5_npix256_merged.h5py


# correction for subjects 1 and 2